# **Procesamiento de Lenguaje Natural**

## Maestría en Inteligencia Artificial Aplicada
#### Tecnológico de Monterrey
#### Prof Luis Eduardo Falcón Morales

### **Adtividad en Equipos Semanas 7 y 8 : LDA y LMM audio-a-texto**

* **Nombres y matrículas:**

  *   Joel Arturo Becerril Balderas — A01797427
  *   Diego Villegas Juarez — A01797347
  *   Camilo José López Amaya — A01797343
  *   Manuel Alejandro Ambriz Baca — A01686824

* **Número de Equipo:** —

* ##### **En cada ejercicio pueden importar los paquetes o librerías que requieran.**

* ##### **En cada ejercicio pueden incluir las celdas y líneas de código que deseen.**

# **Ejercicio 1:**

* #### **Liga de los audios de las fábulas de Esopo:** https://www.gutenberg.org/ebooks/21144

* #### **Descargar los 10 archivos de audio solicitados: 1, 4, 5, 6, 14, 22, 24, 25, 26, 27.**


In [4]:
import os
import requests
from pathlib import Path

# Instalación de dependencias
!pip install groq gensim nltk --quiet

AUDIO_NUMBERS = [1, 4, 5, 6, 14, 22, 24, 25, 26, 27]
AUDIO_DIR = Path("audios")
AUDIO_DIR.mkdir(exist_ok=True)

BASE_URL = "https://www.gutenberg.org/files/21144/mp3/21144-{:02d}.mp3"

print("Descargando audios de las Fábulas de Esopo — Proyecto Gutenberg\n")

for num in AUDIO_NUMBERS:
    url = BASE_URL.format(num)
    filename = AUDIO_DIR / f"fabula_{num:02d}.mp3"
    if filename.exists():
        print(f"  Fábula {num:02d}: ya existe ({filename.stat().st_size/1024:.1f} KB)")
        continue
    print(f"  Descargando fábula {num:02d}... ", end="", flush=True)
    try:
        r = requests.get(url, stream=True, timeout=60)
        r.raise_for_status()
        with open(filename, "wb") as f:
            for chunk in r.iter_content(chunk_size=8192):
                f.write(chunk)
        print(f"OK ({filename.stat().st_size/1024:.1f} KB)")
    except Exception as e:
        print(f"ERROR: {e}")

downloaded = sorted(AUDIO_DIR.glob("*.mp3"))
print(f"\nTotal archivos MP3 disponibles: {len(downloaded)}")
for f in downloaded:
    print(f"  {f.name}  —  {f.stat().st_size/1024:.1f} KB")

Descargando audios de las Fábulas de Esopo — Proyecto Gutenberg

  Descargando fábula 01... OK (1136.2 KB)
  Descargando fábula 04... OK (1052.5 KB)
  Descargando fábula 05... OK (914.2 KB)
  Descargando fábula 06... OK (1122.8 KB)
  Descargando fábula 14... OK (910.3 KB)
  Descargando fábula 22... OK (896.3 KB)
  Descargando fábula 24... OK (1010.2 KB)
  Descargando fábula 25... OK (884.2 KB)
  Descargando fábula 26... OK (996.3 KB)
  Descargando fábula 27... OK (944.3 KB)

Total archivos MP3 disponibles: 10
  fabula_01.mp3  —  1136.2 KB
  fabula_04.mp3  —  1052.5 KB
  fabula_05.mp3  —  914.2 KB
  fabula_06.mp3  —  1122.8 KB
  fabula_14.mp3  —  910.3 KB
  fabula_22.mp3  —  896.3 KB
  fabula_24.mp3  —  1010.2 KB
  fabula_25.mp3  —  884.2 KB
  fabula_26.mp3  —  996.3 KB
  fabula_27.mp3  —  944.3 KB


# **Ejercicio 2a:**

* #### **Comenten el por qué del modelo seleccionado para extracción del texto de los audios.**

* #### **Extraer el contenido de los audios en texto.**

* #### **Sugerencia:** pueden extraerlo en un formato de diccionario, clave:valor $→$ {audio01:fabula01, ...}

### Justificación del modelo: Whisper local (`openai-whisper`)

Seleccionamos **Whisper-small** de OpenAI ejecutado localmente, sin API de pago.

| Criterio | Whisper local (`whisper-small`) | Alternativa (whisper-1 API) |
|---|---|---|
| **Costo** | $0 — corre en tu hardware | ~$0.006 / minuto de audio |
| **Privacidad** | Total — el audio no sale de tu máquina | Audio enviado a OpenAI |
| **Calidad español** | Alta — entrenado con 680k horas, 97 idiomas | Alta |
| **Timestamps** | ✅ Incluidos | Solo con `verbose_json` |

**¿Por qué `small`?** Balance óptimo calidad/velocidad. `whisper-tiny` es más rápido pero comete más errores en español; `whisper-large` requiere GPU para ser práctico.

In [5]:
import whisper
import json
from pathlib import Path

AUDIO_NUMBERS = [1, 4, 5, 6, 14, 22, 24, 25, 26, 27]
AUDIO_DIR = Path("audios")
RAW_JSON = Path("fabulas_raw.json")

# Si ya existe el JSON (re-ejecución del notebook), cargamos directamente
if RAW_JSON.exists():
    with open(RAW_JSON, encoding="utf-8") as f:
        fabulas_raw = json.load(f)
    print(f"Transcripciones cargadas desde {RAW_JSON} ({len(fabulas_raw)} fábulas)\n")
else:
    model = whisper.load_model("small")

    fabulas_raw = {}
    print("Transcribiendo con Whisper-small (local, sin costo, sin API key)\n")

    for num in AUDIO_NUMBERS:
        filename = AUDIO_DIR / f"fabula_{num:02d}.mp3"
        key = f"audio{num:02d}"
        print(f"  {key}: ", end="", flush=True)
        try:
            result = model.transcribe(str(filename), language="es", fp16=False)
            fabulas_raw[key] = result["text"].strip()
            print(f"OK  ({len(result['text'])} chars)")
        except Exception as e:
            print(f"ERROR: {e}")

    with open(RAW_JSON, "w", encoding="utf-8") as f:
        json.dump(fabulas_raw, f, ensure_ascii=False, indent=2)
    print(f"\nGuardado en {RAW_JSON}")

for key in sorted(fabulas_raw):
    print(f"\n[{key}]  {fabulas_raw[key][:180]}...")

100%|███████████████████████████████████████| 461M/461M [00:43<00:00, 11.1MiB/s]


Transcribiendo con Whisper-small (local, sin costo, sin API key)

  audio01: OK  (587 chars)
  audio04: OK  (772 chars)
  audio05: OK  (700 chars)
  audio06: OK  (828 chars)
  audio14: OK  (564 chars)
  audio22: OK  (565 chars)
  audio24: OK  (748 chars)
  audio25: OK  (482 chars)
  audio26: OK  (627 chars)
  audio27: OK  (591 chars)

Guardado en fabulas_raw.json

[audio01]  Las fábulas de Sopo, grabado para LibriVox.org por Paulino, www.paulino.info. Fábula n. 61, el lobo y el cordero en el templo. Tándose cuenta de que era perseguido por un lobo, un ...

[audio04]  Las fábulas de SOFO grabado para LibriVox.org por Roberto Antonio Muñoz, fábula número 64, el lobo y la grúda. A un lobo que comía un hueso, se le atragantó el hueso en la garganta...

[audio05]  Las Fábulas de Sopo, grabado para LibriVox.org por Karen Savage. Fábula número 65. El Lobo y el Caballo. Pasaba un lobo por un sembrado de cebada, pero como no era comida de su gus...

[audio06]  Las fábulas de Esopo, grabado para

# **Ejercicio 2b:**

* #### **Eliminar el inicio y final comunes de los textos extraídos de cada fábula.**

* #### **Sugerencia:** Pueden guardar esta información en un archivo tipo JSON, para que al estar probando diferentes opciones en los ejercicios siguientes, puedan recuperar rápidamente la información de cada video/fábula.

In [6]:
import re
import json

# Inicio común: "Las fábulas de Esopo... Fábula número N."
# Whisper transcribe el número como dígito o como palabra → patrón flexible
PATTERN_INICIO = re.compile(
    r'^.*?f[aá]bula\s+n[uú]mero\s+[\w\s]+?[.,]\s*',
    re.IGNORECASE | re.DOTALL
)
# Final común: "Fin de (la) fábula. Esta es una grabación del dominio público."
PATTERN_FIN = re.compile(
    r'\s*[Ff]in\s+de\s+(?:la\s+)?f[áa]bula[\s\S]*$',
    re.IGNORECASE
)

fabulas_clean = {}

print(f"{'Fábula':<12} {'Original (chars)':>17} {'Limpio (chars)':>15}")
print("-" * 48)

for key in sorted(fabulas_raw):
    text = fabulas_raw[key]
    text_clean = PATTERN_INICIO.sub("", text).strip()
    text_clean = PATTERN_FIN.sub("", text_clean).strip()
    fabulas_clean[key] = text_clean
    print(f"  {key:<10} {len(text):>17} {len(text_clean):>15}")

with open("fabulas.json", "w", encoding="utf-8") as f:
    json.dump(fabulas_clean, f, ensure_ascii=False, indent=2)

print("\nGuardado en fabulas.json")
for key in sorted(fabulas_clean):
    print(f"\n  [{key}]  {fabulas_clean[key][:120]}...")

Fábula        Original (chars)  Limpio (chars)
------------------------------------------------
  audio01                  586             525
  audio04                  771             627
  audio05                  699             563
  audio06                  827             672
  audio14                  563             405
  audio22                  564             389
  audio24                  747             612
  audio25                  481             362
  audio26                  626             489
  audio27                  590             412

Guardado en fabulas.json

  [audio01]  Las fábulas de Sopo, grabado para LibriVox.org por Paulino, www.paulino.info. Fábula n. 61, el lobo y el cordero en el t...

  [audio04]  el lobo y la grúda. A un lobo que comía un hueso, se le atragantó el hueso en la garganta y corría por todas partes en b...

  [audio05]  El Lobo y el Caballo. Pasaba un lobo por un sembrado de cebada, pero como no era comida de su gusto, la dejó y siguió 

# **Ejercicio 3:**

* #### **Apliquen el proceso de limpieza que consideren adecuado.**

* #### **Justifiquen los pasos de limpieza utilizados. Tomen en cuenta que el texto extraído de cada fábula es relativamente pequeño.**

* #### **En caso de que decidan no aplicar esta etapa de limpieza, deberán justificarlo.**

### Estrategia de limpieza

Adoptamos una **limpieza conservadora** dado que cada fábula tiene ~150–200 palabras:

| Paso | Acción | Justificación |
|------|--------|---------------|
| 1 | Minúsculas | Normalización |
| 2 | Eliminar puntuación | No aporta información léxica para LDA |
| 3 | Eliminar dígitos | Residuos de transcripción |
| 4 | Tokenizar | Unidades léxicas |
| 5 | Eliminar stopwords ES | Palabras funcionales sin carga semántica |
| 6 | Filtrar tokens < 3 chars | Residuos no capturados por stopwords |

**Sin stemming ni lematización**: con ~100 tokens por fábula, colapsar formas distintas eliminaría información semántica valiosa. Sin umbral de frecuencia: con 10 documentos, casi todo el vocabulario es 'poco frecuente'.

In [7]:
import re
import json
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

nltk.download("stopwords", quiet=True)
nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)

if "fabulas_clean" not in dir() or not fabulas_clean:
    with open("fabulas.json", encoding="utf-8") as f:
        fabulas_clean = json.load(f)

STOP_ES = set(stopwords.words("spanish"))
MIN_LEN = 3


def limpiar(texto: str) -> list:
    texto = texto.lower()
    texto = re.sub(r"[^\w\s]", " ", texto)
    texto = re.sub(r"\d+", " ", texto)
    tokens = word_tokenize(texto, language="spanish")
    return [t for t in tokens if t.isalpha() and len(t) >= MIN_LEN and t not in STOP_ES]


fabulas_tokens = {k: limpiar(v) for k, v in fabulas_clean.items()}

print(f"{'Fábula':<12} {'Palabras orig':>13} {'Tokens limpios':>15}")
print("-" * 44)
for key in sorted(fabulas_tokens):
    orig = len(fabulas_clean[key].split())
    tok = len(fabulas_tokens[key])
    print(f"  {key:<10} {orig:>13} {tok:>15}")

print("\nMuestra de tokens (primeros 12):")
for key in sorted(fabulas_tokens):
    print(f"  {key}: {fabulas_tokens[key][:12]}")

Fábula       Palabras orig  Tokens limpios
--------------------------------------------
  audio01               93              50
  audio04              119              63
  audio05              105              48
  audio06              125              55
  audio14               73              32
  audio22               72              36
  audio24              119              53
  audio25               67              33
  audio26               88              42
  audio27               72              37

Muestra de tokens (primeros 12):
  audio01: ['fábulas', 'sopo', 'grabado', 'librivox', 'org', 'paulino', 'www', 'paulino', 'info', 'fábula', 'lobo', 'cordero']
  audio04: ['lobo', 'grúda', 'lobo', 'comía', 'hueso', 'atragantó', 'hueso', 'garganta', 'corría', 'todas', 'partes', 'busca']
  audio05: ['lobo', 'caballo', 'pasaba', 'lobo', 'sembrado', 'cebada', 'comida', 'gusto', 'dejó', 'siguió', 'camino', 'encontró']
  audio06: ['lobo', 'asno', 'lobo', 'elegido', 'rey', 'congénere

# **Ejercicio 4:**

In [8]:
import re
from collections import Counter
from gensim import corpora, models

# LDA aplicado por fábula con 1 tópico y 20 palabras clave.
# Dado que cada fábula tiene ~100 tokens, segmentamos en oraciones
# y tratamos cada oración como documento independiente para mejorar
# la calidad de las co-ocurrencias que calcula LDA.

NUM_WORDS = 20
fabulas_keywords = {}

print(f"=== LDA — 1 tópico, {NUM_WORDS} palabras clave por fábula ===\n")

for key in sorted(fabulas_clean):
    text = fabulas_clean[key]

    sentences = [s.strip() for s in re.split(r"[.!?]+", text) if s.strip()]
    sent_tokens = [limpiar(s) for s in sentences]
    sent_tokens = [t for t in sent_tokens if len(t) >= 2]

    if len(sent_tokens) < 3:
        sent_tokens = [fabulas_tokens[key]]

    if not sent_tokens or not any(sent_tokens):
        counter = Counter(fabulas_tokens[key])
        fabulas_keywords[key] = [w for w, _ in counter.most_common(NUM_WORDS)]
        print(f"  {key} (frecuencia): {', '.join(fabulas_keywords[key])}\n")
        continue

    fable_dict = corpora.Dictionary(sent_tokens)
    fable_corpus = [fable_dict.doc2bow(doc) for doc in sent_tokens]

    lda = models.LdaModel(
        corpus=fable_corpus,
        id2word=fable_dict,
        num_topics=1,
        passes=50,
        random_state=42,
        minimum_probability=0.0
    )

    fabulas_keywords[key] = [w for w, _ in lda.show_topic(0, topn=NUM_WORDS)]
    print(f"  {key} ({len(sent_tokens)} oraciones):")
    print(f"    {', '.join(fabulas_keywords[key])}\n")

print(f"Palabras clave extraídas para {len(fabulas_keywords)} fábulas.")

=== LDA — 1 tópico, 20 palabras clave por fábula ===

  audio01 (8 oraciones):
    lobo, ser, cordero, templo, dios, así, mejor, replicó, llamó, remedio, colmillos, perecer, encontraba, prefiero, sacrificados, mayor, vale, tener, sacrificador, honor

  audio04 (7 oraciones):
    lobo, paga, hueso, grulla, cabeza, boca, garganta, pidió, sano, salvo, sana, corruptos, salva, haber, oye, nunca, dijo, crees, sacado, malvados

  audio05 (5 oraciones):
    cebada, caballo, lobo, lobos, preferido, debe, malvado, bueno, creérsela, oídos, oír, estómago, complacer, comieran, amigo, vez, ruido, rato, aunque, sino

  audio06 (7 oraciones):
    lobo, asno, ley, repártelo, tener, llegas, poder, confundido, dijo, llevándola, decretado, comunidad, orejas, moviendo, magnífica, idea, escuchó, escondido, propias, primero

  audio14 (5 oraciones):
    lobo, cabrito, sino, encuentras, ampliamente, insultarle, comenzó, serenamente, replicó, insultando, arrogante, burlándose, enfrentamiento, lugar, menudo, oc

# **Ejercicio 5a y 5b:**

* #### **5a: Mediante el LLM que hayan seleccionado, generar un único enunciado que describa o resuma cada fábula.**

* #### **5b: Mediante el LLM que hayan seleccionado, generar tres posibles enunciados diferentes relacionados con la historia de la fábula.**

* #### **Sugerencia:** En realidad los dos incisos a y b se pueden obtener con un solo prompt que solicite la información y el formato correspondiente para cada una de estas partes. Por ejemplo, para cada fábula la salida puede ser un primer enunciado genérico que resume o describe dicha temática; seguido de tres enunciados, cada uno hablando sobre una situación o parte diferente de la fábula.

### Modelo LLM: Llama 3.3 70B vía Groq

Usamos **Groq** (empresa independiente, chip LPU propio) con **Llama 3.3 70B Versatile**.

| Criterio | Llama 3.3 70B (Groq) | GPT-4o (OpenAI) |
|---|---|---|
| **Costo** | Gratis (free tier) | ~$5 / 1M tokens |
| **Velocidad** | ~500 tok/seg (chip LPU) | ~100 tok/seg |
| **Calidad español** | Alta | Alta |



In [10]:
import os
import json
import getpass
from groq import Groq

# API key gratuita en console.groq.com — getpass oculta la entrada
api_key = os.environ.get("GROQ_API_KEY") or getpass.getpass("GROQ_API_KEY: ")
client = Groq(api_key=api_key)

MODEL = "llama-3.3-70b-versatile"


def analizar_fabula(keywords: list, texto: str) -> dict:
    prompt = f"""Eres un experto en literatura clásica y fábulas de Esopo.

Palabras clave (LDA): {', '.join(keywords)}
Texto: {texto[:500]}

Responde ÚNICAMENTE en este formato:
RESUMEN: [un enunciado de máximo 30 palabras que describa la fábula]
SUBTEMA_1: [situación específica de la historia, máximo 25 palabras]
SUBTEMA_2: [otra situación diferente, máximo 25 palabras]
SUBTEMA_3: [una tercera situación diferente, máximo 25 palabras]

Responde en español."""

    response = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": prompt}],
        temperature=0.7,
        max_tokens=400
    )
    raw = response.choices[0].message.content
    result = {"resumen": "", "subtemas": [], "raw": raw}
    for line in raw.strip().split("\n"):
        line = line.strip()
        if line.startswith("RESUMEN:"):
            result["resumen"] = line[8:].strip()
        elif line.startswith("SUBTEMA_"):
            result["subtemas"].append(":".join(line.split(":")[1:]).strip())
    return result


print(f"=== Ejercicio 5: Groq / {MODEL} ===\n")
fabulas_llm = {}

for key in sorted(fabulas_keywords):
    print(f"Procesando {key}...", flush=True)
    res = analizar_fabula(fabulas_keywords[key], fabulas_clean.get(key, ""))
    fabulas_llm[key] = res
    print(f"  5a) {res['resumen']}")
    for i, s in enumerate(res["subtemas"], 1):
        print(f"  5b.{i}) {s}")
    print()

with open("fabulas_llm.json", "w", encoding="utf-8") as f:
    json.dump(fabulas_llm, f, ensure_ascii=False, indent=2)
print("Guardado en fabulas_llm.json")

=== Ejercicio 5: Groq / llama-3.3-70b-versatile ===

Procesando audio01...
  5a) Un cordero elige ser sacrificado a un dios que ser devorado por un lobo.
  5b.1) Cordero huye de un lobo y se refugia en un templo.
  5b.2) Lobo intenta engañar al cordero para que salga del templo.
  5b.3) Cordero prefiere ser sacrificado a un dios que ser víctima del lobo.

Procesando audio04...
  5a) Lobo y grulla en situación de auxilio y paga.
  5b.1) Lobo con hueso atragantado busca ayuda.
  5b.2) Grulla salva al lobo con astucia.
  5b.3) Lobo no paga, considera la vida paga suficiente.

Procesando audio05...
  5a) Un lobo y un caballo discuten sobre la cebada.
  5b.1) Lobo encuentra cebada, pero no la come.
  5b.2) Lobo intenta engañar al caballo sobre la cebada.
  5b.3) Caballo responde con ironía al lobo malvado.

Procesando audio06...
  5a) Lobo rey impone ley para compartir presas, pero es hipócrita.
  5b.1) Lobo es elegido rey y decreta ley para compartir presas.
  5b.2) Asno descubre hipocresí

# **Ejercicio 6:**

* #### **Incluyan sus conclusiones de la actividad audio-a-texto:**

---

## Conclusiones

### 1. Transcripción de audio (Whisper-small local)

Whisper-small produjo transcripciones de alta calidad en los 10 audios. La especificación de `language="es"` fue determinante: sin ella, Whisper puede confundir el acento latinoamericano del narrador con otros idiomas. Al correr localmente, el audio nunca sale de la máquina (privacidad garantizada) y el costo es cero. La transcripción se guarda en `fabulas_raw.json` para no repetirla al reiniciar el kernel.

### 2. Eliminación de intro/outro (Ejercicio 2b)

El reto fue que Whisper transcribe los números de fábula de forma inconsistente: "fábula número catorce" vs "fábula número 14". Los patrones regex con `[\w\s]+?` resuelven ambos casos. Se verificó manualmente la correcta eliminación en todos los audios.

### 3. Limpieza de texto (Ejercicio 3)

La estrategia conservadora fue la correcta: con ~100-150 tokens por fábula, aplicar stemming o umbrales de frecuencia habría dejado documentos demasiado escasos para LDA.

### 4. LDA — modelado de temas (Ejercicio 4)

LDA está diseñado para corpus grandes; aplicarlo a textos de 100 tokens produce resultados inestables. La estrategia de segmentar cada fábula en oraciones (tratando cada oración como documento independiente) mejoró la coherencia de las palabras clave. Para corpus cortos, alternativas como KeyBERT o YAKE serían más adecuadas.

### 5. LLM — Groq con Llama 3.3 70B (Ejercicio 5)

Llama 3.3 70B vía Groq produjo resúmenes coherentes incluso cuando las palabras clave de LDA eran poco específicas. La velocidad del chip LPU (~500 tok/seg) completó los 10 análisis en segundos. El prompt con formato fijo facilitó el parsing automático.

### Reflexión final

El stack **Whisper local + Groq gratuito** demostró ser viable: costo cero, privacidad total en ASR, y calidad suficiente para esta actividad. La calidad de cada etapa condiciona directamente la siguiente — una mala transcripción produce keywords de LDA irrelevantes, y keywords irrelevantes producen resúmenes genéricos del LLM.

# **Fin de la actividad LDA y LMM: audio-a-texto**